In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")
df["Native ID"] = df["Native ID"].astype("string")
df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")


# Select rows where ISSUE contains "Attribution" (case-sensitive)
df_empty = df[df["ISSUE"].isna()]

df_empty_clean = df_empty[['Network ID',  'Native ID', 'Station ID', 'Unique Names', 'Unique Location (String)'
]].reset_index(drop=True)



df_has_arrow = df_empty_clean[
    df_empty_clean.astype(str).apply(
        lambda row: row.str.contains("->", na=False)
    ).any(axis=1)
].reset_index(drop=True)

df_has_arrow

,Network ID,Native ID,Station ID,Unique Names,Unique Location (String)
0,2,36191,12477,Line Creek,"114.8905 W, 49.88374 N, Elev. 1246 m -> 114.89..."
1,2,37093,11025,Quartz Creek,"117.35992 W, 51.47846 N, Elev. 1065 m -> 117.3..."
2,12,66,1652,BIG SILVER,"121.8596 W, 49.691 N, Elev. 8 m -> 121.8596W, ..."
3,12,440,1976,FIRESIDE,"127.3349 W, 59.7227 N, Elev. 731 m -> 127.335 ..."
4,12,839,2272,ZZ MUCHALAT LAKE,"126.1455 W, 49.8875 N, Elev. 1125 m -> 126.145..."
5,12,903,2298,CARROTT LAKE,"124.4403 W, 53.4 N, Elev. 1000 m ->124.4389 W,..."
6,12,1003,10992,ZZ BAY ST TESTQD,"123.3749333 W, 48.4348 N, Elev. 50 m -> 123.37..."
7,12,1310,12327,ZZ BUCHANAN QD2,"116.9583167 W, 49.97163333 N, Elev. 1784 m -> ..."
8,12,1790,13000,-> CARIBOO CREEK,"0 W, null N, Elev. null m -> 117.1757 W, 51.17..."
9,9,E295949 -> 452,12139,Fraser Lake Endako Mines_60,"125.095833 W, 54.034444 N, Elev. 1005 m"


### update native_id

In [5]:
df_native_id_up = df_has_arrow[df_has_arrow['Native ID'].str.contains("->", na=False)][['Station ID', 'Network ID', 'Native ID',]]



split_ids = df_native_id_up['Native ID'].astype(str).str.split('->', expand=True)

df_native_id_up['old_native_id'] = split_ids[0].str.strip()
df_native_id_up['new_native_id'] = split_ids[1].str.strip()


df_native_id_up

,Station ID,Network ID,Native ID,old_native_id,new_native_id
9,12139,9,E295949 -> 452,E295949,452
10,2588,9,M120001 -> 380,M120001,380
11,2590,9,E256894 -> 356,E256894,356
13,12570,9,Quesnel CP -> M111073 -> 153,Quesnel CP,M111073
14,1574,9,M111073 -> 153,M111073,153
15,2606,9,M111065 -> 155,M111065,155
16,1570,9,E226268 -> 390,E226268,390
17,12288,9,E292249 ->435,E292249,435
27,1596,11,'-> MF1,',MF1
28,1594,11,'-> Nonda1,',Nonda1


In [6]:

check_sql = sa.text("""
SELECT station_id, native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.connect() as conn:
    for _, row in df_native_id_up.iterrows():
        res = conn.execute(
            check_sql,
            {"station_id": int(row["Station ID"])}
        ).fetchone()

        if res is None:
            print(f"❌ Station {row['Station ID']} not found")
            continue

        db_native_id = res.native_id

        if db_native_id == row["old_native_id"]:
            print(
                f"✅ OK to update station {res.station_id}: "
                f"{db_native_id} → {row['new_native_id']}"
            )
        else:
            print(
                f"⚠️ Mismatch for station {res.station_id}: "
                f"DB={db_native_id}, expected={row['old_native_id']}"
            )

✅ OK to update station 12139: E295949 → 452
✅ OK to update station 2588: M120001 → 380
✅ OK to update station 2590: E256894 → 356
✅ OK to update station 12570: Quesnel CP → M111073
✅ OK to update station 1574: M111073 → 153
✅ OK to update station 2606: M111065 → 155
✅ OK to update station 1570: E226268 → 390
✅ OK to update station 12288: E292249 → 435
⚠️ Mismatch for station 1596: DB=MF1, expected='
⚠️ Mismatch for station 1594: DB=Nonda1, expected='


In [7]:
update_station_sql = sa.text("""
UPDATE meta_station
SET native_id = :new_native_id
WHERE station_id = :station_id
  AND native_id = :old_native_id
""")

with engine.begin() as conn:
    for _, row in df_native_id_up.iterrows():
        result = conn.execute(
            update_station_sql,
            {
                "station_id": int(row["Station ID"]),
                "old_native_id": row["old_native_id"],
                "new_native_id": row["new_native_id"],
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {row['Station ID']}: "
                f"{row['old_native_id']} → {row['new_native_id']}"
            )
        else:
            # check why it failed
            res = conn.execute(
                check_sql,
                {"station_id": int(row["Station ID"])}
            ).fetchone()

            if res is None:
                print(f"❌ Station {row['Station ID']} not found")
            else:
                print(
                    f"⚠️ Skipped station {row['Station ID']}: "
                    f"DB={res.native_id}, expected={row['old_native_id']}"
                )

✅ Updated station 12139: E295949 → 452
✅ Updated station 2588: M120001 → 380
✅ Updated station 2590: E256894 → 356
✅ Updated station 12570: Quesnel CP → M111073
✅ Updated station 1574: M111073 → 153
✅ Updated station 2606: M111065 → 155
✅ Updated station 1570: E226268 → 390
✅ Updated station 12288: E292249 → 435
⚠️ Skipped station 1596: DB=MF1, expected='
⚠️ Skipped station 1594: DB=Nonda1, expected='


In [8]:
verify_sql = sa.text("""
SELECT native_id
FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_native_id_up.iterrows():
        # verify
        res = conn.execute(
            verify_sql,
            {"station_id": int(row["Station ID"])}
        ).fetchone()

        if res and res.native_id == row["new_native_id"]:
            print(
                f"✅ Verified: station {row['Station ID']} "
                f"native_id = {res.native_id}"
            )
        else:
            print(
                f"⚠️ Verification failed for station {row['Station ID']}: "
                f"DB={res.native_id if res else None}, "
                f"expected={row['new_native_id']}"
            )

✅ Verified: station 12139 native_id = 452
✅ Verified: station 2588 native_id = 380
✅ Verified: station 2590 native_id = 356
✅ Verified: station 12570 native_id = M111073
✅ Verified: station 1574 native_id = 153
✅ Verified: station 2606 native_id = 155
✅ Verified: station 1570 native_id = 390
✅ Verified: station 12288 native_id = 435
✅ Verified: station 1596 native_id = MF1
✅ Verified: station 1594 native_id = Nonda1


### update station name

In [9]:
df_staion_name_up = df_has_arrow[df_has_arrow['Unique Names'].str.contains("->", na=False)][['Station ID', 'Network ID', 'Unique Names',]]



split_ids = df_staion_name_up['Unique Names'].astype(str).str.split('->', expand=True)

df_staion_name_up['old_name'] = split_ids[0].str.strip()
df_staion_name_up['new_name'] = split_ids[1].str.strip()


df_staion_name_up

,Station ID,Network ID,Unique Names,old_name,new_name
8,13000,12,-> CARIBOO CREEK,,CARIBOO CREEK
12,12545,9,NA -> Fort St John North Camp C,NA,Fort St John North Camp C
13,12570,9,NA -> Quesnel CP,NA,Quesnel CP
18,2470,5,Horn Ck -> Horn Creek,Horn Ck,Horn Creek
19,2503,5,St. Leon Ck -> St. Leon Creek,St. Leon Ck,St. Leon Creek
20,2540,14,Celista Mountain -> Celista,Celista Mountain,Celista
22,2533,14,Spuzzum Creek -> Spuzzum,Spuzzum Creek,Spuzzum
23,2560,14,Upper Mosley Creek -> Mosley Creek Upper,Upper Mosley Creek,Mosley Creek Upper
24,2561,14,Upper Squamish River -> Squamish River Upper,Upper Squamish River,Squamish River Upper
25,12439,11,Coalmine Wx -> CoalmineWx,Coalmine Wx,CoalmineWx


In [10]:
check_sql = sa.text("""
SELECT
    station_id,
    station_name
FROM meta_history
WHERE station_id = :station_id
""")

import numpy as np

with engine.begin() as conn:
    for _, row in df_staion_name_up.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != old_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {old_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name}, the names match "
            )


⚠️ Station 13000 (DB, XLS values differ)
   DB : station name is None
   XLS: station name is 
✅ Station 12545, NA, the names match 
✅ Station 12570, NA, the names match 
✅ Station 2470, Horn Ck, the names match 
✅ Station 2503, St. Leon Ck, the names match 
✅ Station 2540, Celista Mountain, the names match 
✅ Station 2533, Spuzzum Creek, the names match 
✅ Station 2560, Upper Mosley Creek, the names match 
✅ Station 2561, Upper Squamish River, the names match 
⚠️ Station 12439 (DB, XLS values differ)
   DB : station name is CoalmineWx
   XLS: station name is Coalmine Wx
⚠️ Station 12433 (DB, XLS values differ)
   DB : station name is Gnat Pass Stn
   XLS: station name is Gnat Pass


In [11]:
update_sql = sa.text("""
UPDATE meta_history
SET
    station_name  = :new_name
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_staion_name_up.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_name": new_name,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_name}) → "
                f"({new_name})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 13000: () → (CARIBOO CREEK)
✅ Updated station 12545: (NA) → (Fort St John North Camp C)
✅ Updated station 12570: (NA) → (Quesnel CP)
✅ Updated station 2470: (Horn Ck) → (Horn Creek)
✅ Updated station 2503: (St. Leon Ck) → (St. Leon Creek)
✅ Updated station 2540: (Celista Mountain) → (Celista)
✅ Updated station 2533: (Spuzzum Creek) → (Spuzzum)
✅ Updated station 2560: (Upper Mosley Creek) → (Mosley Creek Upper)
✅ Updated station 2561: (Upper Squamish River) → (Squamish River Upper)
✅ Updated station 12439: (Coalmine Wx) → (CoalmineWx)
✅ Updated station 12433: (Gnat Pass) → (Gnat Pass Stn)


In [12]:
import numpy as np

with engine.begin() as conn:
    for _, row in df_staion_name_up.iterrows():

        station_id = int(row["Station ID"])

        old_name  = row["old_name"]
        new_name  = row["new_name"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---

        if db_row.station_name != new_name:
            print(
                f"⚠️ Station {station_id} (DB, XLS values differ)\n"
                f"   DB : station name is {db_row.station_name}\n"
                f"   XLS: station name is {new_name}"
            )
        else:
            print(
                f"✅ Station {station_id}, {db_row.station_name} the names match "
            )


✅ Station 13000, CARIBOO CREEK the names match 
✅ Station 12545, Fort St John North Camp C the names match 
✅ Station 12570, Quesnel CP the names match 
✅ Station 2470, Horn Creek the names match 
✅ Station 2503, St. Leon Creek the names match 
✅ Station 2540, Celista the names match 
✅ Station 2533, Spuzzum the names match 
✅ Station 2560, Mosley Creek Upper the names match 
✅ Station 2561, Squamish River Upper the names match 
✅ Station 12439, CoalmineWx the names match 
✅ Station 12433, Gnat Pass Stn the names match 


In [13]:
df_loc_up = df_has_arrow[df_has_arrow['Unique Location (String)'].str.contains("->", na=False)][['Station ID', 'Network ID', 'Unique Location (String)',]]


split_ids = df_loc_up['Unique Location (String)'].astype(str).str.split('->', expand=True)



df_loc_up

,Station ID,Network ID,Unique Location (String)
0,12477,2,"114.8905 W, 49.88374 N, Elev. 1246 m -> 114.89..."
1,11025,2,"117.35992 W, 51.47846 N, Elev. 1065 m -> 117.3..."
2,1652,12,"121.8596 W, 49.691 N, Elev. 8 m -> 121.8596W, ..."
3,1976,12,"127.3349 W, 59.7227 N, Elev. 731 m -> 127.335 ..."
4,2272,12,"126.1455 W, 49.8875 N, Elev. 1125 m -> 126.145..."
5,2298,12,"124.4403 W, 53.4 N, Elev. 1000 m ->124.4389 W,..."
6,10992,12,"123.3749333 W, 48.4348 N, Elev. 50 m -> 123.37..."
7,12327,12,"116.9583167 W, 49.97163333 N, Elev. 1784 m -> ..."
8,13000,12,"0 W, null N, Elev. null m -> 117.1757 W, 51.17..."
21,2539,14,"118.617 W, 50.45 N, Elev. 1890 m -> 118.6185 W..."


In [14]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df_loc_up[cols] = df_loc_up['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)
# 
df_loc_up = df_loc_up.drop(columns=['Unique Location (String)'])


df_loc_up = df_loc_up.dropna(
    subset=['new_lat', 'new_lon', 'new_elev'],
    how='all'
)

df_loc_up


,Station ID,Network ID,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev
1,11025,2,51.478460,-117.359920,1065.0,51.47995,-117.35812,1068.0
2,1652,12,49.691000,-121.859600,8.0,49.69100,-121.85960,152.0
3,1976,12,59.722700,-127.334900,731.0,59.72260,-127.33500,735.0
4,2272,12,49.887500,-126.145500,1125.0,49.88750,-126.14540,1125.0
5,2298,12,53.400000,-124.440300,1000.0,53.40080,-124.43890,1000.0
7,12327,12,49.971633,-116.958317,1784.0,50.04940,-117.08860,1784.0
21,2539,14,50.450000,-118.617000,1890.0,50.44950,-118.61850,1890.0


In [15]:
check_sql = sa.text("""
SELECT
    station_id,
    lat,
    lon,
    elev
FROM meta_history
WHERE station_id = :station_id
""")


with engine.begin() as conn:
    for _, row in df_loc_up.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 2. Compare with expected OLD values ---
        def norm(x):
            if pd.isna(x):
                return None
            return round(float(x), 6)

        def same(a, b):
            return norm(a) == norm(b)

        if not (
            same(db_row.lat,  old_lat) and
            same(db_row.lon,  old_lon) and
            same(db_row.elev, old_elev)
        ):
            print(
                f"⚠️ Station {station_id} skipped (DB values differ)\n"
                f"   DB : lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}\n"
                f"   XLS: lat={old_lat}, lon={old_lon}, elev={old_elev}"
            )
            continue

        # --- 3. Values match ---
        print(
            f"✅ Station {station_id} match confirmed\n"
            f"   lat={db_row.lat}, lon={db_row.lon}, elev={db_row.elev}"
        )


✅ Station 11025 match confirmed
   lat=51.47846, lon=-117.35992, elev=1065
✅ Station 1652 match confirmed
   lat=49.691, lon=-121.8596, elev=8
✅ Station 1976 match confirmed
   lat=59.7227, lon=-127.3349, elev=731
✅ Station 2272 match confirmed
   lat=49.8875, lon=-126.1455, elev=1125
✅ Station 2298 match confirmed
   lat=53.4, lon=-124.4403, elev=1000
✅ Station 12327 match confirmed
   lat=49.97163333, lon=-116.9583167, elev=1784
✅ Station 2539 match confirmed
   lat=50.45, lon=-118.617, elev=1890


In [16]:
import numpy as np

update_sql = sa.text("""
UPDATE meta_history
SET
    lat  = :new_lat,
    lon  = :new_lon,
    elev = :new_elev
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for _, row in df_loc_up.iterrows():

        station_id = int(row["Station ID"])

        old_lat  = row["old_lat"]
        old_lon  = row["old_lon"]
        old_elev = row["old_elev"]

        new_lat  = row["new_lat"]
        new_lon  = row["new_lon"]
        new_elev = row["new_elev"]

        # --- 1. Read current DB values ---
        db_row = conn.execute(
            check_sql,
            {"station_id": station_id}
        ).fetchone()

        if db_row is None:
            print(f"❌ Station {station_id} not found in DB")
            continue

        # --- 3. Perform update ---
        result = conn.execute(
            update_sql,
            {
                "station_id": station_id,
                "new_lat": new_lat,
                "new_lon": new_lon,
                "new_elev": new_elev,
            }
        )

        if result.rowcount == 1:
            print(
                f"✅ Updated station {station_id}: "
                f"({old_lat}, {old_lon}, {old_elev}) → "
                f"({new_lat}, {new_lon}, {new_elev})"
            )
        else:
            print(f"⚠️ Station {station_id}: no update applied")

✅ Updated station 11025: (51.47846, -117.35992, 1065.0) → (51.47995, -117.35812, 1068.0)
✅ Updated station 1652: (49.691, -121.8596, 8.0) → (49.691, -121.8596, 152.0)
✅ Updated station 1976: (59.7227, -127.3349, 731.0) → (59.7226, -127.335, 735.0)
✅ Updated station 2272: (49.8875, -126.1455, 1125.0) → (49.8875, -126.1454, 1125.0)
✅ Updated station 2298: (53.4, -124.4403, 1000.0) → (53.4008, -124.4389, 1000.0)
✅ Updated station 12327: (49.97163333, -116.9583167, 1784.0) → (50.0494, -117.0886, 1784.0)
✅ Updated station 2539: (50.45, -118.617, 1890.0) → (50.4495, -118.6185, 1890.0)
